# V74 - Borda rank fusion

Pre-registered plan: all tested combinations blend scores, but rank-space fusion is a different operation and scale-invariant. Borda count over five views (word, assessment, mentions, content, popularity) faces word-only scores under the standard grouped folds and fresh seed. Same locks.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics.pairwise import cosine_similarity

SEED = 616
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v74")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## View OOFs, Borda ranks, gate

Each view ranks modules per user. Borda points sum across views. Negative total rank orders the final list.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
low = [d.lower() for d in docs]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
STOP = set("dengan yang dan di ke dari untuk pada adalah ini itu sebagai dalam juga akan ada".split())
MN = np.zeros((len(all_users), len(MODS)))
for j, m in enumerate(MODS):
    row = catalog.set_index("module_id").loc[m] if m in set(catalog["module_id"]) else None
    keys = set(re.findall(r"[a-z]+", (str(row["module_name"]) + " " + str(row["description_and_syllabus"])).lower())) - STOP if row is not None else set()
    keys = {k for k in keys if len(k) > 3}
    for i, d in enumerate(low):
        MN[i, j] = sum(d.count(k) for k in keys)
MN = (MN - MN[:n_train].mean(0)) / (MN[:n_train].std(0) + 1e-9)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1).fit(docs + catalog["description_and_syllabus"].fillna("").tolist())
syll = [catalog.set_index("module_id").loc[m, "description_and_syllabus"] if m in set(catalog["module_id"]) else "" for m in MODS]
CM = cosine_similarity(vec.transform(docs), vec.transform(syll))
VIEWS = {"word": W, "assess": A, "mentions": np.nan_to_num(MN), "content": CM}

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(Xtr, alpha=1.0):
    oof = np.zeros_like(Y)
    fids = np.zeros(n_train, int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
        fids[va] = fi + 1
    return oof, fids

oof = {"popularity": np.tile(Y.mean(0), (n_train, 1))}
for k, V in VIEWS.items():
    oof[k], fid = ridge_oof(V[:n_train])
    print(f"{k}: {ndcg5(Y, oof[k]).mean():.5f}")
borda = np.zeros_like(Y)
for k, S in oof.items():
    ranks = np.argsort(np.argsort(-S, axis=1), axis=1)
    borda += len(MODS) - 1 - ranks
base = ndcg5(Y, oof["word"])
d = ndcg5(Y, -borda) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fid == k], -borda[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"borda OOF={ndcg5(Y, -borda).mean():.5f} baseline={base.mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT Borda" if (lo > GATE_FLOOR and wf >= 0) else "KEEP scores; rank fusion adds nothing")

word: 0.60915
assess: 0.42156


mentions: 0.53565
content: 0.42408


borda OOF=0.12102 baseline=0.60915 delta=-0.48813 CI95=[-0.49638,-0.47983] worst-fold=-0.52411
KEEP scores; rank fusion adds nothing
